In [1]:
# Import the libraries and define paths required for demand forecasting

import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")

FEATURES_PATH = PROCESSED_DATA_DIR / "pricing_features.parquet"
MODEL_DATA_PATH = PROCESSED_DATA_DIR / "forecasting_data.parquet"

In [2]:
# Load the prepared feature dataset used for demand forecasting

df = pd.read_parquet(FEATURES_PATH)

df = df.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

display(df.head())

Dataset shape: (13563568, 30)
Date range: 2011-02-26 00:00:00 to 2016-05-22 00:00:00


,date,item_id,dept_id,cat_id,store_id,state_id,wm_yr_wk,weekday,wday,month,...,week_of_year,is_weekend,snap_active,lag_1_demand,lag_7_demand,rolling_7d_demand,rolling_28d_demand,previous_price,price_change_pct,price_changed
0,2011-02-26,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11105,Saturday,1,2,...,8,1,0,0.0,1.0,0.857143,1.428571,2.0,0.0,0
1,2011-02-27,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11105,Sunday,2,2,...,8,1,0,6.0,3.0,1.571429,1.607143,2.0,0.0,0
2,2011-02-28,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11105,Monday,3,2,...,9,0,0,6.0,2.0,2.000000,1.750000,2.0,0.0,0
3,2011-03-01,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11105,Tuesday,4,3,...,9,0,1,1.0,0.0,1.857143,1.750000,2.0,0.0,0
4,2011-03-02,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11105,Wednesday,5,3,...,9,0,1,1.0,0.0,2.000000,1.750000,2.0,0.0,0


In [3]:
# Select the features and target required for demand forecasting

feature_cols = [
    "sell_price",
    "wday",
    "month",
    "year",
    "week_of_year",
    "is_weekend",
    "snap_active",
    "lag_1_demand",
    "lag_7_demand",
    "rolling_7d_demand",
    "rolling_28d_demand",
    "previous_price",
    "price_change_pct",
    "price_changed"
]

categorical_cols = [
    "item_id",
    "store_id",
    "cat_id"
]

target_col = "demand"

model_cols = (
    ["date"]
    + categorical_cols
    + feature_cols
    + [target_col]
)

model_df = df[model_cols].copy()

print("Modeling rows:", len(model_df))
print("Numerical features:", len(feature_cols))
print("Categorical features:", len(categorical_cols))
print("Target:", target_col)

Modeling rows: 13563568
Numerical features: 14
Categorical features: 3
Target: demand


In [4]:
# Split chronologically so future observations never leak into model training

max_date = model_df["date"].max()

test_start = max_date - pd.Timedelta(days=27)
val_start = test_start - pd.Timedelta(days=28)

train_df = model_df[model_df["date"] < val_start].copy()
val_df = model_df[
    (model_df["date"] >= val_start) &
    (model_df["date"] < test_start)
].copy()
test_df = model_df[model_df["date"] >= test_start].copy()

print("Train:", train_df["date"].min(), "to", train_df["date"].max(),
      "| Rows:", len(train_df))

print("Validation:", val_df["date"].min(), "to", val_df["date"].max(),
      "| Rows:", len(val_df))

print("Test:", test_df["date"].min(), "to", test_df["date"].max(),
      "| Rows:", len(test_df))

Train: 2011-02-26 00:00:00 to 2016-03-27 00:00:00 | Rows: 13142056
Validation: 2016-03-28 00:00:00 to 2016-04-24 00:00:00 | Rows: 210756
Test: 2016-04-25 00:00:00 to 2016-05-22 00:00:00 | Rows: 210756


In [7]:
# Evaluate a 7-day seasonal baseline before training machine-learning models

from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_pred = val_df["lag_7_demand"]
baseline_actual = val_df["demand"]

baseline_mae = mean_absolute_error(
    baseline_actual,
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_actual,
        baseline_pred
    )
)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Baseline RMSE: {baseline_rmse:.4f}")

Baseline MAE: 1.7570
Baseline RMSE: 3.3836


In [8]:
# Prepare training, validation and test matrices for model training

model_features = categorical_cols + feature_cols

X_train = train_df[model_features]
y_train = train_df[target_col]

X_val = val_df[model_features]
y_val = val_df[target_col]

X_test = test_df[model_features]
y_test = test_df[target_col]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (13142056, 17)
X_val: (210756, 17)
X_test: (210756, 17)

y_train: (13142056,)
y_val: (210756,)
y_test: (210756,)


In [9]:
# Install LightGBM in the notebook environment

import sys
!{sys.executable} -m pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)


In [10]:
# Convert categorical columns to category dtype for efficient LightGBM training

for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_val[col] = X_val[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print("Categorical columns:")
print(X_train[categorical_cols].dtypes)

Categorical columns:
item_id     category
store_id    category
cat_id      category
dtype: object


In [11]:
# Train the first LightGBM demand forecasting model

import lightgbm as lgb

model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_cols,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50)
    ],
    eval_set=[(X_val, y_val)],
    eval_metric="rmse"
)

c:\Users\aayus\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.631754 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2960
[LightGBM] [Info] Number of data points in the train set: 13142056, number of used features: 17
[LightGBM] [Info] Start training from score 0.856861
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.77929	valid_0's poisson: -1.24866
[100]	valid_0's rmse: 2.50433	valid_0's poisson: -1.3666
[150]	valid_0's rmse: 2.47656	valid_0's poisson: -1.38549
[200]	valid_0's rmse: 2.47083	valid_0's poisson: -1.39021
[250]	valid_0's rmse: 2.46581	valid_0's poisson: -1.39282
[300]	valid

,num_leaves,63
,learning_rate,0.05
,n_estimators,500
,objective,'poisson'
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000


In [12]:
# Evaluate the trained LightGBM model against the validation dataset

val_pred = model.predict(X_val)

val_pred = np.maximum(val_pred, 0)

lgb_mae = mean_absolute_error(y_val, val_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

rmse_improvement = (
    (baseline_rmse - lgb_rmse) / baseline_rmse
) * 100

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"LightGBM MAE: {lgb_mae:.4f}")

print(f"\nBaseline RMSE: {baseline_rmse:.4f}")
print(f"LightGBM RMSE: {lgb_rmse:.4f}")

print(f"\nRMSE improvement over baseline: {rmse_improvement:.2f} %")

Baseline MAE: 1.7570
LightGBM MAE: 1.3613

Baseline RMSE: 3.3836
LightGBM RMSE: 2.4512

RMSE improvement over baseline: 27.56 %


In [13]:
# Identify which features contribute most to the demand forecast

feature_importance = pd.DataFrame({
    "feature": model_features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(feature_importance)

,feature,importance
0,item_id,13279
1,rolling_28d_demand,2535
2,wday,2433
3,rolling_7d_demand,2008
4,lag_1_demand,1951
5,store_id,1607
6,week_of_year,1531
7,sell_price,1046
8,lag_7_demand,844
9,year,780


In [14]:
# Evaluate the selected LightGBM model on the final unseen test period

test_pred = model.predict(X_test)
test_pred = np.maximum(test_pred, 0)

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

test_baseline_pred = test_df["lag_7_demand"]

test_baseline_mae = mean_absolute_error(
    y_test,
    test_baseline_pred
)

test_baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_baseline_pred
    )
)

print(f"Test Baseline MAE: {test_baseline_mae:.4f}")
print(f"LightGBM Test MAE: {test_mae:.4f}")

print(f"\nTest Baseline RMSE: {test_baseline_rmse:.4f}")
print(f"LightGBM Test RMSE: {test_rmse:.4f}")

print(
    f"\nTest RMSE improvement: "
    f"{((test_baseline_rmse - test_rmse) / test_baseline_rmse) * 100:.2f} %"
)

Test Baseline MAE: 1.8519
LightGBM Test MAE: 1.4207

Test Baseline RMSE: 3.5531
LightGBM Test RMSE: 2.5506

Test RMSE improvement: 28.21 %


In [15]:
# Save the trained demand forecasting model for later prediction and deployment

import joblib

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODELS_DIR / "demand_forecast_lgbm.pkl"

joblib.dump(model, model_path)

print("Model saved to:", model_path)
print("Model features:", len(model_features))

Model saved to: ..\models\demand_forecast_lgbm.pkl
Model features: 17
